In [ ]:
# Week 3 Day 6: Building a Reliable Evaluation Pipeline

## Today's Goal

Today I will combine preprocessing, modeling, cross-validation, and multiple evaluation metrics into one reliable sklearn workflow.

## Learning Objectives

- Understand Pipeline as an end-to-end modeling workflow
- Use cross_validate() to evaluate multiple metrics
- Compare training and validation performance
- Interpret mean CV scores and standard deviations
- Review how Pipeline prevents preprocessing leakage
- Build a clean and reusable model evaluation workflow

## Expected Output

By the end of today, I should be able to:

1. Build a StandardScaler + Logistic Regression Pipeline
2. Evaluate the complete Pipeline with Stratified K-Fold
3. Calculate accuracy, precision, recall, F1, and 

In [1]:
## Dataset Preparation
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np

# Load the Breast Cancer Wisconsin dataset
data = load_breast_cancer()

X = data.data

## Original sklearn labels:
# 0 = maglignant
# 1 = benign
#
# we reserve them so that:
# 1 = malignant
# 0 = benign
y = 1 - data.target

print("Full X shape:", X.shape)
print("Full y shape:", y.shape)

print("\nClass counts:")
print("Benign (0):", np.sum(y == 0))
print("Malignant (1):", np.sum(y == 1))

# Hold out the final test set
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nDevelopment set:", X_dev.shape)
print("Final test set:", X_test.shape)

Full X shape: (569, 30)
Full y shape: (569,)

Class counts:
Benign (0): 357
Malignant (1): 212

Development set: (455, 30)
Final test set: (114, 30)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg', LogisticRegression(max_iter=1000, random_state=42))])


In [3]:
## Cross-Validation with Multiple Metrics
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define stratified 5-fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate several classification metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(
    pipeline,
    X_dev,
    y_dev,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

In [4]:
print(cv_results.keys())

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'train_accuracy', 'test_precision', 'train_precision', 'test_recall', 'train_recall', 'test_f1', 'train_f1', 'test_roc_auc', 'train_roc_auc'])


In [5]:
print("Validation accuracy scores:")
print(cv_results["test_accuracy"])

print("\nNumber of folds:")
print(len(cv_results["test_accuracy"]))

Validation accuracy scores:
[0.95604396 0.96703297 1.         0.97802198 0.96703297]

Number of folds:
5


In [ ]:
## Cross-Validation Summary
import numpy as np
import pandas as pd

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

summary_rows = []

for metric in metrics:
    train_scores = cv_results[f"train_{metric}"]
    val_scores = cv_results[f"test_{metric}"]

    summary_rows.append({
        "Metric": metric,
        "Train Mean": np.mean(train_scores),
        "Validation Mean": np.mean(val_scores),
        "Validation Std": np.std(val_scores),
        "Train-Val Gap": np.mean(train_scores) - np.mean(val_scores)
    })

summary_df = pd.DataFrame(summary_rows)

print(summary_df)

In [6]:
## Cross-Validation Summary
import numpy as np
import pandas as pd

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

summary_rows = []

for metric in metrics:
    train_scores = cv_results[f"train_{metric}"]
    val_scores = cv_results[f"test_{metric}"]

    summary_rows.append({
        "Metric": metric,
        "Train Mean": np.mean(train_scores),
        "Validation": np.mean(val_scores),
        "Validation Std": np.std(val_scores),
        "Train-Val Gap": np.mean(train_scores) - np.mean(val_scores)
    })

summary_df = pd.DataFrame(summary_rows)

print(summary_df)

      Metric  Train Mean  Validation  Validation Std  Train-Val Gap
0   accuracy    0.987912    0.973626        0.014906       0.014286
1  precision    0.996981    0.977143        0.027994       0.019838
2     recall    0.970588    0.952941        0.039896       0.017647
3         f1    0.983599    0.964048        0.020700       0.019551
4    roc_auc    0.997691    0.995769        0.004666       0.001922


In [ ]:
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall   : {test_recall:.4f}")
print(f"F1       : {test_f1:.4f}")
print(f"ROC-AUC  : {test_auc:.4f}")

In [8]:
## Final Test Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Fit the complete pipeline using all development data
pipeline.fit(X_dev, y_dev)

# Class predictions
y_test_pred = pipeline.predict(X_test)

# Probability for Class 1 = malignant
y_test_prob = pipeline.predict_proba(X_test)[:1]

# Final test metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_pred)

print("Final Test Results")
print("------------------")
print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall : {test_recall:.4f}")
print(f"F1 : {test_f1:.4f}")
print(f"ROC-AUC : {test_auc:.4f}")

Final Test Results
------------------
Accuracy : 0.9649
Precision : 0.9750
Recall : 0.9286
F1 : 0.9512
ROC-AUC : 0.9573


In [ ]:
# Reflection

1. Why is Pipeline important when preprocessing is used together with cross-validation?
Because Pipeline shows more detailed metrics in the whole dataset from training set to test set which suggests more infromation to evaluate
the model.
2. What extra information did cross_validate() give us compared with looking at accuracy alone?
The all metrics are shown like accuracy, precision, roc_auc and f1 in k-Folds stage.
3. After today's experiment, why should we compare CV results and final test results instead of expecting them to be exactly the same?
Because for different sets or samples, the model likely has slightly different performance in different stages like CV and final test. 
Sometimes, overfitting likely appears in CV. We should compare the results to evaluate a model better.